# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [7]:
# Write your code below.

from dotenv import load_dotenv
load_dotenv()

True

In [8]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [9]:
import os
from glob import glob

# Write your code below.

PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [12]:
ddf = dd.read_parquet(parquet_files)
print("Columns in the DataFrame:", ddf.columns)

Columns in the DataFrame: Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year'],
      dtype='object')


In [15]:
# Write your code below.

ddf = dd.read_parquet(parquet_files)

dd_feat = ddf.assign(
    Close_lag_1=ddf["Close"].shift(1),
    Adj_Close_lag_1=ddf["Adj Close"].shift(1)
)

dd_feat = dd_feat.assign(
    returns=(dd_feat["Close"] / dd_feat["Close_lag_1"]) - 1,
    hi_lo_range=dd_feat["High"] - dd_feat["Low"]
)

print("Columns in dd_feat:", dd_feat.columns)

Columns in dd_feat: Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'returns',
       'hi_lo_range'],
      dtype='object')


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [16]:
# Write your code below.
pdf = dd_feat.compute()
pdf["returns_ma_10"] = pdf["returns"].rolling(window=10).mean()

# It was not necessary to convert to pds to calculate the moving average
# It would be better to do it in Dask for large datasets 

print("Columns:", pdf.columns)


Columns: Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'returns',
       'hi_lo_range', 'returns_ma_10'],
      dtype='object')


In [17]:
type(pdf)

pandas.core.frame.DataFrame

In [18]:
pdf[["returns", "returns_ma_10"]].head(15)


,returns,returns_ma_10
38739,NaN,NaN
38740,-0.082386,NaN
38741,0.089783,NaN
38742,-0.090909,NaN
38743,0.026563,NaN
38744,0.003044,NaN
38745,0.022762,NaN
38746,0.001484,NaN
38747,0.017778,NaN
38748,0.027656,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.